# Dashboard de Análise de E-commerce

Este projeto tem como objetivo desenvolver uma aplicação interativa utilizando Dash e Plotly para apresentar visualmente os principais indicadores de um conjunto de dados de e-commerce.

Os dados utilizados foram previamente tratados e estão disponíveis no arquivo `ecommerce_estatistica.csv`. A aplicação reúne diferentes tipos de visualização, permitindo ao usuário analisar preços, avaliações, marcas, categorias e a relação entre avaliações e quantidade vendida sem a necessidade de interagir diretamente com o código Python.

### Tecnologias utilizadas
- Python
- Pandas
- NumPy
- Plotly
- Dash
- Scikit-learn

## 1. Importação das bibliotecas

Nesta etapa são importadas as bibliotecas necessárias para leitura e manipulação dos dados, criação das visualizações, construção do modelo de regressão linear e desenvolvimento da aplicação Dash.

In [41]:
from pathlib import Path

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from sklearn.linear_model import LinearRegression

from dash import Dash, dcc, html

## 2. Leitura e preparação dos dados

O arquivo `ecommerce_estatistica.csv` é carregado em um DataFrame do Pandas. A coluna `Unnamed: 0`, criada anteriormente como índice durante a exportação do arquivo, é removida por não possuir valor analítico para esta etapa.

As primeiras linhas do DataFrame são exibidas para verificar se os dados foram carregados corretamente.

### Verificação dos dados

Antes da criação das visualizações, é realizada uma verificação básica da dimensão do conjunto de dados e da existência de valores ausentes.

In [42]:
from pathlib import Path

BASE_DIR = Path.cwd()
arquivo = BASE_DIR / 'ecommerce_estatistica.csv'

df = pd.read_csv(arquivo)

df = df.drop(columns=['Unnamed: 0'], errors='ignore')

display(df.head())

print(f'Quantidade de registros: {df.shape[0]}')
print(f'Quantidade de colunas: {df.shape[1]}')
print(f'Valores ausentes: {df.isnull().sum().sum()}')

,Título,Nota,N_Avaliações,Desconto,Marca,Material,Gênero,Temporada,Review1,Review2,...,Nota_MinMax,N_Avaliações_MinMax,Desconto_MinMax,Preço_MinMax,Marca_Cod,Material_Cod,Temporada_Cod,Qtd_Vendidos_Cod,Marca_Freq,Material_Freq
0,Kit 10 Cuecas Boxer Lupo Cueca Box Algodão Mas...,4.5,3034.0,18.0,lupo,algodão,Masculino,outono/inverno,As cuecas são boas; porém você percebe na cost...,"Pelo preço promocional, com ctz tem algum defe...",...,0.814815,0.334178,0.213115,0.378585,463,25,3,10000.0,0.042292,0.176444
1,Kit Com 10 Cuecas Boxer Algodão Sem Costura Zo...,4.7,5682.0,20.0,zorba,algodão,Masculino,não definido,O tecido é bom e são confortáveis. Só que a nu...,Vendo comentários de outros consumidores vejo ...,...,0.888889,0.625937,0.245902,0.322329,838,25,1,50000.0,0.009095,0.176444
2,Kit 10 Cuecas Boxer Mash Algodão Cotton Box Or...,4.6,1700.0,22.0,mash,algodão,Masculino,primavera/verão,"As cuecas são boas, porém meu marido usa g e p...","E o tamanho certo, mas em baixo dela, fica mui...",...,0.851852,0.187197,0.278689,0.372617,494,25,7,10000.0,0.010914,0.176444
3,Kit 3 Short Jeans Feminino Cintura Alta Barato...,4.4,507.0,9.0,menina linda,jean,Feminino,primavera/verão,Estou encantada com essas peças!.\nOs shorts s...,"Recomendo, tecido confortável, igual a foto.",...,0.777778,0.055751,0.065574,0.201767,509,74,7,1000.0,0.010005,0.025466
4,Blusa + Calça Térmica Treino Futebol Criança I...,4.7,58.0,5.0,roupa zero grau,termico unissex,Sem gênero infantil,outono/inverno,"Produto ótimo , mesmo após várias lavagens não...",Produto de boa qualidade.\nNão gruda pêlos e n...,...,0.888889,0.006280,0.000000,0.114508,669,166,3,100.0,0.002274,0.000910


Quantidade de registros: 295
Quantidade de colunas: 23
Valores ausentes: 0


## 3. Criação das visualizações

Os gráficos do projeto anterior foram recriados utilizando Plotly para permitir interação do usuário dentro da aplicação Dash.

### 3.1 Histograma — Distribuição dos Produtos por Faixa de Preço

O histograma permite observar como os produtos estão distribuídos entre as diferentes faixas de preço e identificar onde ocorre a maior concentração de itens.

In [43]:
fig_histograma = px.histogram(
    df,
    x='Preço',
    nbins=15,
    title='Distribuição dos Produtos por Faixa de Preço',
    labels={
        'Preço': 'Preço dos Produtos (R$)',
        'count': 'Quantidade de Produtos'
    }
)

fig_histograma.update_layout(
    xaxis_title='Preço dos Produtos (R$)',
    yaxis_title='Quantidade de Produtos'
)

### 3.2 Gráfico de Dispersão — Preço e Nota

O gráfico de dispersão é utilizado para analisar a relação entre duas variáveis numéricas. Neste caso, são comparados o preço e a nota dos produtos para verificar se produtos mais caros apresentam avaliações superiores.

In [44]:
fig_dispersao = px.scatter(
    df,
    x='Preço',
    y='Nota',
    title='Relação entre Preço e Nota dos Produtos',
    labels={
        'Preço': 'Preço (R$)',
        'Nota': 'Nota'
    }
)

fig_dispersao.update_layout(
    xaxis_title='Preço (R$)',
    yaxis_title='Nota'
)

### 3.3 Mapa de Calor — Correlação entre Variáveis

O mapa de calor apresenta a matriz de correlação das principais variáveis numéricas. Essa visualização facilita a identificação de relações positivas ou negativas entre os dados analisados.

In [45]:
colunas_numericas = [
    'Nota',
    'N_Avaliações',
    'Desconto',
    'Preço',
    'Qtd_Vendidos_Cod'
]

correlacao = df[colunas_numericas].corr().round(2)

fig_heatmap = px.imshow(
    correlacao,
    text_auto=True,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Correlação entre as Variáveis Numéricas'
)

fig_heatmap.update_layout(
    xaxis_title='Variáveis analisadas',
    yaxis_title='Variáveis analisadas'
)

### 3.4 Gráfico de Barras — Marcas com Maior Presença

O gráfico de barras apresenta as dez marcas com maior quantidade de produtos no conjunto de dados, facilitando a comparação entre as categorias.

In [46]:
top_marcas = (
    df['Marca']
    .value_counts()
    .head(10)
    .reset_index()
)

top_marcas.columns = ['Marca', 'Quantidade']

fig_barras = px.bar(
    top_marcas,
    x='Marca',
    y='Quantidade',
    title='10 Marcas com Maior Quantidade de Produtos',
    labels={
        'Marca': 'Marca',
        'Quantidade': 'Quantidade de Produtos'
    }
)

fig_barras.update_layout(
    xaxis_title='Marca',
    yaxis_title='Quantidade de Produtos'
)

### 3.5 Gráfico de Pizza — Distribuição por Gênero

O gráfico de pizza apresenta a participação das diferentes categorias de gênero no conjunto de produtos. Categorias com pouca ocorrência são agrupadas como "Outros" para melhorar a leitura da visualização.

In [47]:
genero = df['Gênero'].value_counts()

principais = genero[genero >= 5].copy()
outros = genero[genero < 5].sum()

if outros > 0:
    principais['Outros'] = outros

df_genero = principais.reset_index()
df_genero.columns = ['Gênero', 'Quantidade']

fig_pizza = px.pie(
    df_genero,
    names='Gênero',
    values='Quantidade',
    title='Distribuição dos Produtos por Gênero'
)

fig_pizza.update_traces(
    textposition='inside',
    textinfo='percent+label'
)

### 3.6 Gráfico de Densidade — Notas dos Produtos

O gráfico de densidade permite visualizar em quais valores as notas dos produtos estão mais concentradas, fornecendo uma visão suavizada da distribuição das avaliações.

In [48]:
import plotly.figure_factory as ff

notas = df['Nota'].dropna()

fig_densidade = ff.create_distplot(
    [notas],
    ['Notas'],
    show_hist=False,
    show_rug=False
)

fig_densidade.update_layout(
    title='Concentração das Notas dos Produtos',
    xaxis_title='Nota dos Produtos',
    yaxis_title='Densidade Estimada'
)

### 3.7 Regressão Linear — Avaliações e Quantidade Vendida

A regressão linear é utilizada para visualizar a tendência existente entre o número de avaliações e a quantidade vendida. O modelo é criado com Scikit-learn e a linha de regressão é adicionada ao gráfico de dispersão utilizando Plotly.

Essa análise permite observar a associação entre as duas variáveis, sem afirmar necessariamente uma relação de causa e efeito.

In [49]:
from sklearn.linear_model import LinearRegression

X = df[['N_Avaliações']]
y = df['Qtd_Vendidos_Cod']

modelo_regressao = LinearRegression()
modelo_regressao.fit(X, y)

x_linha = np.linspace(
    df['N_Avaliações'].min(),
    df['N_Avaliações'].max(),
    100
)

x_previsao = pd.DataFrame({
    'N_Avaliações': x_linha
})

y_linha = modelo_regressao.predict(x_previsao)

fig_regressao = px.scatter(
    df,
    x='N_Avaliações',
    y='Qtd_Vendidos_Cod',
    title='Tendência entre Número de Avaliações e Quantidade Vendida',
    labels={
        'N_Avaliações': 'Número de Avaliações',
        'Qtd_Vendidos_Cod': 'Quantidade Vendida'
    }
)

fig_regressao.add_trace(
    go.Scatter(
        x=x_linha,
        y=y_linha,
        mode='lines',
        name='Regressão Linear'
    )
)

fig_regressao.update_layout(
    xaxis_title='Número de Avaliações',
    yaxis_title='Quantidade Vendida'
)

## 4. Construção da aplicação Dash

Após a criação das visualizações, é criada uma aplicação Dash responsável por reunir os gráficos em uma única interface.

O componente `html.Div` é utilizado como estrutura principal da página, enquanto cada gráfico Plotly é inserido utilizando `dcc.Graph`. Também são aplicadas configurações básicas de estilo para melhorar a organização e a legibilidade do dashboard.

Dessa forma, o usuário final pode explorar as visualizações por meio de uma interface web sem precisar executar ou modificar diretamente o código Python.

In [50]:
app = Dash(__name__)

app.layout = html.Div(

    style={
        'maxWidth': '1200px',
        'margin': 'auto',
        'fontFamily': 'Arial',
        'padding': '20px'
    },

    children=[

        html.H1(
            'Dashboard de Análise de E-commerce',
            style={
                'textAlign': 'center',
                'marginBottom': '10px'
            }
        ),

        html.P(
            'Visualização interativa dos principais indicadores do conjunto de dados de e-commerce.',
            style={
                'textAlign': 'center',
                'marginBottom': '40px'
            }
        ),

        html.H2('Distribuição dos Produtos por Faixa de Preço'),
        dcc.Graph(figure=fig_histograma),

        html.H2('Relação entre Preço e Nota dos Produtos'),
        dcc.Graph(figure=fig_dispersao),

        html.H2('Correlação entre as Variáveis Numéricas'),
        dcc.Graph(figure=fig_heatmap),

        html.H2('10 Marcas com Maior Quantidade de Produtos'),
        dcc.Graph(figure=fig_barras),

        html.H2('Distribuição dos Produtos por Gênero'),
        dcc.Graph(figure=fig_pizza),

        html.H2('Concentração das Notas dos Produtos'),
        dcc.Graph(figure=fig_densidade),

        html.H2(
            'Tendência entre Número de Avaliações e Quantidade Vendida'
        ),
        dcc.Graph(figure=fig_regressao)

    ]
)

app.run(
    jupyter_mode='external',
    debug=False,
    port=8055
)

Dash app running on http://127.0.0.1:8055/


## 5. Execução da aplicação

Por fim, a aplicação Dash é iniciada em um servidor local. O modo `external` permite abrir o dashboard diretamente no navegador, onde as visualizações do Plotly permanecem interativas.

In [51]:
app.run(
    jupyter_mode='external',
    debug=False,
    port=8055
)

Dash app running on http://127.0.0.1:8055/


## Conclusão

O projeto permitiu integrar as etapas de leitura de dados, análise, visualização e desenvolvimento de uma aplicação web utilizando Python.

As visualizações desenvolvidas com Plotly possibilitam explorar diferentes características do conjunto de dados de e-commerce, enquanto o Dash reúne essas informações em uma interface única e interativa.

Com isso, o usuário final consegue acessar os resultados de forma mais simples e intuitiva, sem precisar interagir diretamente com o código Python, atendendo ao objetivo proposto para o projeto.